# 02 · Вектор управления: решение остаётся за студентом

**Цель:** усилить одно поведение агента без обучения — оставлять содержательное решение студенту. Это главное правило продукта и самое частое нарушение: модель формально задаёт вопрос, но заодно выдаёт готовую цель или гипотезу.

Контраст берётся из `data/train/decisions.jsonl`: эталон, где ассистент даёт критерии и вопрос, против плохого ответа, где выдаёт готовую формулировку. Разность средних активаций прибавляется к скрытым состояниям среднего слоя.

Фильтрация нечестных запросов сюда не входит: это отдельная модель перед агентом, и вектор её не заменяет. Теория — `books/03-alignment.pdf`.

In [ ]:
from common import (MODEL_ID, SYSTEM, RUNS, SHOWCASE, load_rows, user_message, evaluate, fmt, table, show_case)

import json
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import memory_report
from vlmkit.steering import SteeringVector, decoder_layers, suggest_layer

golden = load_rows("golden")
train = load_rows("train")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
def report(results, per_row, verdicts=None, ids=SHOWCASE):
    """Полные ответы модели на показательные ситуации с проверками и вердиктом судьи."""
    for i, row in enumerate(golden):
        if row["id"] in ids:
            show_case(row, results[i], per_row[i]["checks"], verdicts[i] if verdicts else None)

before, before_rows, before_summary = evaluate(model, processor, golden)
print(fmt(before_summary))

## Контраст

Берём ситуации из `decisions.jsonl`: студент просит сформулировать за него тему, цель, гипотезу или задачи. Положительный текст — запрос плюс эталон, где ассистент даёт критерии и возвращает решение студенту; отрицательный — тот же запрос плюс готовая формулировка. Общее начало в разности сокращается, остаётся направление «решение за студентом».

In [ ]:
contrast = load_rows("train")
contrast = [r for r in contrast if r["id"].startswith(("TR-TIB", "TR-PB", "TR-RME", "TR-RW-07", "TR-ED-13")) and "refuses" in r["checks"]]
print(len(contrast), "пар для вектора")

layers = decoder_layers(model)
layer = suggest_layer(model)
kind = getattr(layers[layer], "layer_type", getattr(layers[layer], "block_type", "?"))
print(f"слоёв {len(layers)}, берём {layer}, тип {kind}")

vector = SteeringVector.from_contrast(
    model, processor,
    positive=[f"{user_message(r)}\n{r['answer']}" for r in contrast],
    negative=[f"{user_message(r)}\n{r['rejected']}" for r in contrast],
    layer=layer,
)
print(f"норма разности средних {vector.scale:.1f}; strength=1 прибавляет ровно её")

## Развёртка по силе

Единица — сама разность средних. Смотрите на пару recall / FPR и на checks_all: вектор, который поднимает recall ценой FPR или ломает связность ответов, бесполезен.

In [ ]:
sweep = {"база": before_summary}
for s in (0.5, 1.0, 1.5, 2.0):
    with vector.applied(model, strength=s):
        _, _, summary = evaluate(model, processor, golden)
    sweep[f"s={s}"] = summary
    print(f"s={s:<4} {fmt(summary)}")
table(sweep, keys=("skill_acc", "completed", "checks_all", "checks_mean", "block_recall", "decline_fpr", "length"))

## Результат

Лучшая сила — по разности recall − FPR. Ниже четыре ситуации целиком с вектором: среди них одна с недопустимым запросом (обход антиплагиата) и три легитимные — на них ответ не должен превратиться в отказ.

In [ ]:
best = max((s for s in sweep if s != "база"), key=lambda k: sweep[k]["checks_mean"] - sweep[k]["decline_fpr"])
strength = float(best.split("=")[1])
print(f"берём {best}")

with vector.applied(model, strength=strength):
    after, after_rows, after_summary = evaluate(model, processor, golden)
report(after, after_rows)

RUNS.mkdir(exist_ok=True)
vector.save(str(RUNS / "decision-vector.pt"))
(RUNS / "steering.json").write_text(json.dumps({"strength": strength, **after_summary}, ensure_ascii=False, indent=2), encoding="utf-8")

## Наоборот

Тот же вектор с минусом подавляет отказ: recall падает, модель начинает выполнять недопустимые просьбы. Это известный способ снимать отказы с открытых моделей — и причина, по которой одним системным промптом безопасность не обеспечить.

In [ ]:
with vector.applied(model, strength=-strength):
    print(f"s={-strength}:", fmt(evaluate(model, processor, golden)[2]))